# 第 7 集 — Modulation:用 signal 控制参数

⏱ 约 30 分钟 · 难度 ★★★

**这一集控制的音乐元素:movement(动态)** — filter 的开合、wobble 的节奏、pitch 的滑动。

**Max 里的对应 patch:** 把 `adsr~`、`cycle~ 3`、`line~` 的输出接到 `lores~` 的 cutoff inlet 和 `phasor~` 的 frequency inlet。

**你会听到:**
1. filter sweep:声音从闷逐渐打开
2. filter envelope:带 "piu" 感的 pluck bass
3. LFO → cutoff:"wub wub wub",并且对上 BPM
4. tremolo、vibrato、pitch drop
5. 一个完整的 wobble bass

**学完你能做到:**
- 把任何 array 接到 cutoff、音量、wavetable position 上
- 解释 modulation amount 为什么用 "八度" 做单位
- 把 LFO 的速度对上 BPM
- 解释 `freq * t` 在 pitch 变化时为什么是错的,以及 `phasor~` 实际怎么做

## 工具箱

前几集做好的东西,原样搬过来。运行一下就行,不用重读(难的地方都有 `#` 注释)。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

sr = 44100

def play(x, gain=0.3):
    # np.clip 把超过 ±1 的部分压住,防止爆音
    return Audio(np.clip(gain * np.asarray(x), -1, 1), rate=sr, normalize=False)

def scope(x, title="", seconds=0.03):
    n = int(sr * seconds)                      # 只画开头这么多个 sample
    plt.figure(figsize=(9, 2.5))
    plt.plot(np.arange(n) / sr, np.asarray(x)[:n])
    plt.xlabel("time (s)")
    plt.title(title)
    plt.show()

def make_table(harmonic_amps, N=2048):
    p = np.arange(N) / N                       # 表里每个位置的 phase,0 到 1
    table = np.zeros(N)                        # 先做一张全是 0 的空表
    for k, amp in enumerate(harmonic_amps, start=1):
        # 第 k 个 harmonic:在表里走 k 个周期,音量 amp
        table += amp * np.sin(2 * np.pi * k * p)
    return table / np.max(np.abs(table))       # normalize:峰值变成 1

def make_frames(F=16, max_harmonics=32, N=2048):
    frames = []
    for f in range(F):
        # 第 f 张表有几个 harmonic:从 1 个均匀增加到 max_harmonics 个
        n_h = 1 + round(f * (max_harmonics - 1) / (F - 1))
        frames.append(make_table([1 / k for k in range(1, n_h + 1)], N))
    return np.array(frames)                    # 叠成二维 array:F 行 × N 列

def osc_wavetable(frames, phase, pos):
    F, N = frames.shape

    # --- 在哪两张表之间 ---
    fpos = np.clip(pos, 0, 1) * (F - 1)        # 0..1 换算成 0..F-1 的 "表编号",可以带小数
    f0 = np.floor(fpos).astype(int)            # 下面那张表
    f1 = np.minimum(f0 + 1, F - 1)             # 上面那张表(不超过最后一张)
    ffrac = fpos - f0                          # 离下面那张有多远,0..1

    # --- 在表里的哪两个位置之间(和第 3 集一样)---
    p = phase * N
    i0 = np.floor(p).astype(int) % N           # 左边的位置
    i1 = (i0 + 1) % N                          # 右边的位置,到表尾绕回 0
    frac = p - np.floor(p)                     # 小数部分

    # 先在每张表内部插值,再在两张表之间插值
    a = (1 - frac) * frames[f0, i0] + frac * frames[f0, i1]
    b = (1 - frac) * frames[f1, i0] + frac * frames[f1, i1]
    return (1 - ffrac) * a + ffrac * b

def adsr(t, a, d, s, r, gate):
    a, d, r = max(a, 1e-4), max(d, 1e-4), max(r, 1e-4)    # 时间不能是 0,否则下面会除以 0
    # 按住期间:(0,0) → (a,1) → (a+d,s),之后停在 s
    held = np.interp(t, [0, a, a + d], [0, 1, s])
    # 松开那一刻 envelope 有多高(可能还没走完 attack)
    level_at_release = np.interp(gate, [0, a, a + d], [0, 1, s])
    # 从那个高度用 r 秒直线降到 0;np.clip 让它降到 0 后不再变负
    released = level_at_release * np.clip(1 - (t - gate) / r, 0, 1)
    return np.where(t < gate, held, released)  # 松开前用 held,松开后用 released

def lowpass(x, cutoff, res=0.0):
    x = np.asarray(x, dtype=float)
    n_samples = len(x)
    # cutoff 给一个数或一个 array 都行:统一变成和声音一样长的 array
    cutoff = np.clip(np.broadcast_to(cutoff, n_samples), 20, sr * 0.45)

    # --- 把 cutoff / res 换算成 filter 内部用的系数(每个 sample 一组)---
    g = np.tan(np.pi * cutoff / sr)
    k = 1.414 * (1 - res) + 0.05 * res         # damping:res 越大 k 越小,共鸣越强
    a1 = 1 / (1 + g * (g + k))
    a2 = g * a1
    a3 = g * a2

    # .tolist() 把 array 变成普通 list:在循环里逐个取数时快很多
    xs, A1, A2, A3 = x.tolist(), a1.tolist(), a2.tolist(), a3.tolist()
    y = [0.0] * n_samples
    ic1 = ic2 = 0.0                            # 两个状态变量 = gen~ 里的两个 history

    for n in range(n_samples):                 # 一个 sample 一个 sample 按顺序算
        v3 = xs[n] - ic2
        v1 = A1[n] * ic1 + A2[n] * v3
        v2 = ic2 + A2[n] * ic1 + A3[n] * v3    # v2 就是 lowpass 的输出
        ic1 = 2 * v1 - ic1                     # 更新状态,留给下一个 sample 用
        ic2 = 2 * v2 - ic2
        y[n] = v2
    return np.array(y)

frames = make_frames()

freq = 55
dur = 2.0
n_samples = int(sr * dur)
t = np.arange(n_samples) / sr
phase = (freq * t) % 1

saw = osc_wavetable(frames, phase, 1.0)

`saw` 是一个 55 Hz(A1)的低音 saw,2 秒。这一集大部分例子都拿它当原料。

## 1. 回顾:数 vs array

第 4 集的结论:

> 传一个数就是静态的,传一个 array 就是会动的。

`lowpass(x, cutoff, res)` 的 `cutoff` 也一样。在 Max 里这就是 "往 cutoff inlet 接 number" 和 "接 signal" 的区别。

**modulation 的全部内容就是:做一个 array,把它接到某个参数上。** 这一集剩下的部分只是在换不同的 array 和不同的参数。

## 2. Filter sweep(`line~` → cutoff)

先用第 4 集的 `np.linspace`,让 cutoff 从 100 Hz 直线走到 6000 Hz。

**应该听到:** 声音一开始很快变亮,然后后面大半段几乎没什么变化。

In [ ]:
cutoff_linear = np.linspace(100, 6000, n_samples)
play(lowpass(saw, cutoff_linear, res=0.5))

为什么不均匀?因为耳朵按 **倍数**(八度)听频率,不是按 Hz。

- 100 → 200 Hz 是一个八度
- 3000 → 6000 Hz 也是一个八度

直线走的话,第一个八度只用了 100 Hz 的距离(一瞬间就过去了),最后一个八度却占了 3000 Hz 的距离(整整后半段)。

`np.geomspace(100, 6000, n)`:也是从 100 走到 6000,但 **每一步乘同一个数**,而不是加同一个数。这样每个八度花的时间一样长。

**应该听到:** 从闷到亮,全程匀速地打开。

In [ ]:
cutoff_geo = np.geomspace(100, 6000, n_samples)

plt.figure(figsize=(9, 3))
plt.plot(t, cutoff_linear, label="linspace")
plt.plot(t, cutoff_geo, label="geomspace")
plt.xlabel("time (s)")
plt.ylabel("cutoff (Hz)")
plt.legend()
plt.show()

play(lowpass(saw, cutoff_geo, res=0.5))

**记住:** 和频率有关的东西(cutoff、pitch),都按倍数走。

## 3. Filter envelope(`adsr~` → cutoff)

用第 5 集的 `adsr()` 做一条 envelope,让它去推 cutoff。

envelope 的值在 0 到 1 之间。怎么把它变成 Hz?按上面的结论,用倍数:

`cutoff = base * 2 ** (amount * env)`

- `2 ** 八度数` = 升高这么多个八度(`2 ** 1` = 2 倍,`2 ** 2` = 4 倍,`2 ** 3` = 8 倍)
- `env = 0` 时:`2 ** 0 = 1`,cutoff 就是 `base`
- `env = 1` 时:cutoff 比 `base` 高 `amount` 个八度

所以 `base` 是 filter 的 cutoff 旋钮,`amount` 是 envelope amount 旋钮,单位是八度。

In [ ]:
flt_env = adsr(t, a=0.001, d=0.25, s=0.0, r=0.1, gate=1.0)
amp_env = adsr(t, a=0.005, d=0.6, s=0.0, r=0.1, gate=1.0)

base = 150
amount = 5
cutoff = base * 2 ** (amount * flt_env)        # env 0 → 150 Hz;env 1 → 150 × 2^5 = 4800 Hz

plt.figure(figsize=(9, 2.5))
plt.plot(t, cutoff)
plt.xlabel("time (s)")
plt.ylabel("cutoff (Hz)")
plt.show()

play(lowpass(saw, cutoff, res=0.5) * amp_env)

**应该听到:** 一个开头很亮、迅速变闷的 pluck bass,带一点 "piu" 的感觉。

注意这里用了 **两条** envelope:`flt_env` 管 cutoff,`amp_env` 管音量。filter envelope 比 amp envelope 短,所以先变闷、再变小声。

### Amount 的大小

**应该听到:** 三个 pluck。amount 为 1 时几乎没有 "piu";3 时明显;5 时很夸张。

In [ ]:
for amount in [1, 3, 5]:
    print("amount =", amount, "octaves")
    display(play(lowpass(saw, 150 * 2 ** (amount * flt_env), res=0.5) * amp_env))

## 4. LFO → cutoff(`cycle~ 3` → cutoff)

LFO 就是一个很慢的 oscillator。第 1 集的 sine,频率改成 3 Hz 就是了。

`lfo` 在 -1 到 1 之间摆动。套进同一个式子:

`cutoff = 300 * 2 ** (2 * lfo)`

- `lfo = -1` → 300 低 2 个八度 = 75 Hz
- `lfo = 0` → 300 Hz
- `lfo = +1` → 300 高 2 个八度 = 1200 Hz

**应该听到:** "wub wub wub",每秒 3 次。

In [ ]:
lfo_rate = 3
lfo = np.sin(2 * np.pi * lfo_rate * t)         # 和第 1 集的 sine 一样,只是频率很低

cutoff = 300 * 2 ** (2 * lfo)

plt.figure(figsize=(9, 2.5))
plt.plot(t, cutoff)
plt.xlabel("time (s)")
plt.ylabel("cutoff (Hz)")
plt.show()

play(lowpass(saw, cutoff, res=0.6))

### 对上 BPM

LFO 的速度(Hz)= 每秒几次。音乐里更想说的是 "每拍几次"。

- `bpm / 60` = 每秒几拍
- 乘上 "每拍几次" = 每秒几次 = LFO 的 Hz

140 BPM、每拍 2 次(八分音符):`140 / 60 * 2 = 4.67 Hz`。

**应该听到:** wobble 的速度是 140 BPM 的八分音符。2 秒里大约 9 次。

In [ ]:
bpm = 140
per_beat = 2                                   # 每拍 2 次 = 八分音符;改成 4 是十六分,改成 3 是三连音
lfo_rate = bpm / 60 * per_beat
print("LFO rate:", round(lfo_rate, 2), "Hz")

lfo = np.sin(2 * np.pi * lfo_rate * t)
play(lowpass(saw, 300 * 2 ** (2 * lfo), res=0.6))

### LFO 的形状

LFO 是 oscillator,所以第 2 集的做法全都能用:先做一个慢的 phase,再变成不同形状。

- **sine**:平滑地开合 → "wub"
- **saw down**(从 1 掉到 -1):每次突然打开、然后慢慢合上 → "yoi yoi"
- **square**:只有开和关两种状态 → 一亮一暗交替

**应该听到:** 三种不同性格的 wobble,速度相同。

In [ ]:
lfo_phase = (lfo_rate * t) % 1                 # 慢的 phasor~

lfo_shapes = {
    "sine":     np.sin(2 * np.pi * lfo_phase),
    "saw down": 1 - 2 * lfo_phase,                         # 1 → -1
    "square":   np.where(lfo_phase < 0.5, 1.0, -1.0),
}

fig, axes = plt.subplots(3, 1, figsize=(9, 4), sharex=True)
for ax, name in zip(axes, lfo_shapes):
    ax.plot(t, lfo_shapes[name])
    ax.set_ylabel(name)
axes[-1].set_xlabel("time (s)")
plt.show()

for name in lfo_shapes:
    print(name)
    display(play(lowpass(saw, 300 * 2 ** (2 * lfo_shapes[name]), res=0.6)))

## 5. 同一个 LFO,别的参数

modulation 不限于 cutoff。

### 音量 → tremolo

音量不能是负的,所以把 -1..1 的 LFO 挪到 0..1:`0.5 + 0.5 * lfo`。

**应该听到:** 音量以 140 BPM 八分音符的速度一强一弱。

In [ ]:
tremolo = 0.5 + 0.5 * lfo                      # -1..1 → 0..1
play(lowpass(saw, 800) * tremolo)

### Wavetable position

第 4 集做过。**应该听到:** 音色本身在暗和亮之间来回,和 filter wobble 类似但更 "干净",因为没有 resonance。

In [ ]:
play(osc_wavetable(frames, phase, 0.5 + 0.5 * lfo))

## 6. Pitch modulation:这里有个坑

想让 pitch 在 2 秒内从 110 Hz 滑到 220 Hz(升一个八度)。直觉的写法是把 `freq` 换成一个 array:

In [ ]:
freq_glide = np.geomspace(110, 220, n_samples)

wrong_phase = (freq_glide * t) % 1             # 看起来合理,其实是错的
play(np.sin(2 * np.pi * wrong_phase))

**应该听到:** 音高上升得 **明显超过** 一个八度(最后到了 370 Hz 左右,而不是 220 Hz)。这是错的。

**为什么:** `freq * t` 的意思是 "从头到现在 **一直** 是这个频率,所以一共走了这么多个周期"。频率在变的时候,这句话不成立。在第 2 秒,它算的是 "220 × 2 = 440 个周期",好像从一开始就是 220 Hz 一样 — 但实际上前面大部分时间频率都比 220 低,根本走不了那么多。为了凑够这个数,phase 被迫越跑越快。

### `phasor~` 实际上是怎么做的

它不看 "从头到现在",只看 **这一个 sample 该往前走多少**,然后一直累加。

- 一秒走 `freq` 个周期,一秒有 `sr` 个 sample
- 所以 **每个 sample 走 `freq / sr` 个周期**
- 把每一步累加起来,就是 phase

"累加" 是 `np.cumsum`(cumulative sum):第 n 个数 = 前 n 个数的总和。

In [ ]:
steps = np.array([1, 2, 3, 4])
print("steps  =", steps)
print("cumsum =", np.cumsum(steps))            # 1, 1+2, 1+2+3, 1+2+3+4

这也是一种 "记住上一次" — 和上一集的 `history` 是同一个想法。只不过累加太常用了,numpy 提供了现成的 `cumsum`,不用自己写循环。

这叫 **phase accumulation**。

In [ ]:
def phasor(freq, n_samples):
    # freq / sr = 这一个 sample 里 phase 往前走多少
    # cumsum 把每一步累加起来;% 1 只留小数部分
    return np.cumsum(np.broadcast_to(freq, n_samples) / sr) % 1

right_phase = phasor(freq_glide, n_samples)
play(np.sin(2 * np.pi * right_phase))

**应该听到:** 正好升一个八度,停在 220 Hz。

`np.broadcast_to(freq, n_samples)` 的作用:如果 `freq` 是一个数,就把它变成 `n_samples` 个一样的数;如果已经是 array,就原样用。所以 `phasor(110, n)` 和 `phasor(某个array, n)` 都行。

**从现在起,phase 都用 `phasor(freq, n_samples)` 来做。** 之前的 `(freq * t) % 1` 只在 freq 不变时才对。

### Vibrato

pitch 也按倍数走。半音是八度的 1/12,所以:

`freq = 220 * 2 ** (半音数 / 12 * lfo)`

**应该听到:** 一个音高每秒抖动 5 次的音,上下各半个半音,像歌手的颤音。

In [ ]:
vib_lfo = np.sin(2 * np.pi * 5 * t)
freq_vib = 220 * 2 ** (0.5 / 12 * vib_lfo)     # 上下各 0.5 个半音

play(osc_wavetable(frames, phasor(freq_vib, n_samples), 0.8))

### Modulate 一个 modulator

歌手通常不是一开口就颤,而是先平着唱,再慢慢加上颤音。

做法:让 vibrato 的 **深度** 也随时间变化 — 用一条 envelope 去乘 LFO。前 0.5 秒深度是 0,之后 1 秒内升到 1。

**应该听到:** 先是一个平稳的音,然后颤音逐渐出现并加深。

In [ ]:
depth = np.interp(t, [0, 0.5, 1.5], [0, 0, 1])             # 0.5 秒前是 0,1.5 秒时到 1
freq_vib = 220 * 2 ** (0.7 / 12 * vib_lfo * depth)         # LFO × 深度

play(osc_wavetable(frames, phasor(freq_vib, n_samples), 0.8))

### Pitch drop

pitch 从高处(50 Hz 往上 4 个八度 = 800 Hz)快速掉到 50 Hz,配一个短的 amp envelope。

**应该听到:** 一声 "咚 / piu",像电子 kick 或 laser。把 `d=0.12` 改短会更像 kick,改长会更像 laser。

In [ ]:
pitch_env = adsr(t, a=0.001, d=0.12, s=0.0, r=0.1, gate=1.0)
freq_drop = 50 * 2 ** (4 * pitch_env)                      # env 1 → 800 Hz;env 0 → 50 Hz
kick_amp = adsr(t, a=0.001, d=0.5, s=0.0, r=0.1, gate=1.0)

play(np.sin(2 * np.pi * phasor(freq_drop, n_samples)) * kick_amp, gain=0.6)

## 7. 全部接在一起:wobble bass

- oscillator:55 Hz 的 wavetable
- 一个 saw-down LFO(140 BPM 八分音符)同时推 **wavetable position** 和 **cutoff**
- 再加一点点 pitch 上的 LFO,让它更 "脏"

**应该听到:** 一个典型的 "yoi yoi yoi" wobble bass。

In [ ]:
wob = lfo_shapes["saw down"]                                   # -1..1
wob01 = 0.5 + 0.5 * wob                                        # 0..1

bass_phase = phasor(55 * 2 ** (0.3 / 12 * wob), n_samples)     # pitch:上下 0.3 个半音
bass = osc_wavetable(frames, bass_phase, 0.2 + 0.8 * wob01)    # wavetable position:0.2..1
bass = lowpass(bass, 200 * 2 ** (3 * wob01), res=0.7)          # cutoff:200..1600 Hz

plt.figure(figsize=(9, 3.5))
plt.specgram(bass + 1e-9, NFFT=2048, Fs=sr, noverlap=1792, cmap="magma", vmin=-120)
plt.ylim(0, 3000)
plt.xlabel("time (s)")
plt.ylabel("frequency (Hz)")
plt.show()

play(bass, gain=0.2)

spectrogram 里每一个 "鼓包" 就是一次 "yoi":亮的区域突然冲高,再慢慢落回去 — 正是 saw-down LFO 的形状。

## 常见错误

| 现象 | 原因 |
|---|---|
| sweep 听起来不匀速 | 对频率用了 `linspace`,应该用 `geomspace` 或 `base * 2 ** (...)` |
| pitch 滑动的幅度不对 | 用了 `(freq * t) % 1`。freq 会变时必须用 `phasor()` |
| cutoff 变成负数 / 报错 | 用了 `base + amount * lfo`(加法)。用乘法形式就不会变负 |
| tremolo 听起来像失真 | 音量的 LFO 没有挪到 0..1,负的那一半把波形翻转了 |

## 练习

**1.** 把第 4 节的 BPM wobble 改成十六分音符(每拍 4 次),再改成三连音(每拍 3 次)。

<details><summary>答案</summary>

```python
for per_beat in [4, 3]:
    rate = 140 / 60 * per_beat
    l = np.sin(2 * np.pi * rate * t)
    display(play(lowpass(saw, 300 * 2 ** (2 * l), res=0.6)))
```
</details>

**2.** 做一个往 **上** 滑的音:从 110 Hz 在 0.3 秒内升到 440 Hz,然后停住。提示:`np.interp` 做一条 0 → 1 的线,再用 `110 * 2 ** (2 * 线)`。

<details><summary>答案</summary>

```python
rise = np.interp(t, [0, 0.3], [0, 1])
play(np.sin(2 * np.pi * phasor(110 * 2 ** (2 * rise), n_samples)))
```
</details>

**3.** 让 LFO 的 **速度** 本身逐渐加快:从 1 Hz 到 12 Hz。提示:LFO 也是 oscillator,速度会变时它的 phase 也要用 `phasor()`。

<details><summary>答案</summary>

```python
rate = np.geomspace(1, 12, n_samples)
l = np.sin(2 * np.pi * phasor(rate, n_samples))
play(lowpass(saw, 300 * 2 ** (2 * l), res=0.6))
```
wobble 越来越快。
</details>

## 小结

| Max | Python |
|---|---|
| signal 接到 cutoff inlet | `lowpass(x, cutoff_array, res)` |
| `line~`(线性) | `np.linspace(a, b, n)` |
| `line~` → `mtof~`(按音高走) | `np.geomspace(a, b, n)` |
| LFO:`cycle~ 3` | `np.sin(2 * np.pi * 3 * t)` |
| mod amount(八度) | `base * 2 ** (amount * mod)` |
| `phasor~`(signal-rate 频率) | `phasor(freq, n)` = `np.cumsum(freq / sr) % 1` |
| `+=`(gen~)/ `accum` | `np.cumsum(...)` |

- modulation = 做一个 array,接到一个参数上
- 和频率有关的参数按倍数走:`base * 2 ** (八度数)`
- pitch 会变时,phase 必须用累加的方式算

**下一集:** 把 oscillator、filter、envelope、modulation 全部装进一个函数 — 一个完整的 synth。